In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import numpy as np

In [12]:
def yf_download(tickers, start_date, end_date):
    stocks_df = yf.download(tickers, start= start_date, end= end_date, auto_adjust=False)
    # if only one ticker, extract stock column from column tuples
    if len(tickers) == 1:
        stocks_df = stocks_df.xs(tickers[0], axis=1, level=1)
    stocks_df.columns.names = [None] * len(stocks_df.columns.names)
    return(stocks_df)

In [13]:
def last_trading_day(year):
    return (pd.Timestamp(f'{year}-12-31') if pd.Timestamp(f'{year}-12-31').isoweekday() < 6 \
            else pd.Timestamp(f'{year}-12-31') - pd.offsets.BDay(1)).strftime('%Y-%m-%d')

In [14]:
tickers = ['TLT', 'LQD', 'SPY',"^TNX"]
stocks_df = yf_download(tickers,start_date=last_trading_day(2020),end_date="2026-01-01")
stocks1_df = stocks_df["Adj Close"]
stocks1_df.columns=["TLT_ret","LQD_ret","SPY_ret","Rate_change"]
stocks2_df = stocks1_df.pct_change()
stocks2_df

/opt/anaconda3/lib/python3.13/site-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/opt/anaconda3/lib/python3.13/site-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/opt/anaconda3/lib/python3.13/site-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/opt/anaconda3/lib/python3.13/site-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[*********************100%***********************]  4 of 4 completed

,TLT_ret,LQD_ret,SPY_ret,Rate_change
Date,,,,
2020-12-31,NaN,NaN,NaN,NaN
2021-01-04,-0.005068,-0.013614,-0.001204,0.000000
2021-01-05,-0.003202,0.006887,-0.007427,0.041439
2021-01-06,-0.008102,0.005979,-0.020529,0.091100
2021-01-07,-0.000147,0.014857,-0.008814,0.027831
...,...,...,...,...
2025-12-24,0.003901,0.003518,0.006057,-0.007916
2025-12-26,-0.000090,-0.000101,-0.003294,0.000000
2025-12-29,0.001356,-0.003564,0.003761,-0.004836


In [15]:
spread = pd.read_csv("BAA10Y.csv")
stocks2_df.index = pd.to_datetime(stocks2_df.index)
spread["Date"] = pd.to_datetime(spread["observation_date"])
spread = spread.set_index("Date")
spread["Credit_spread"]= spread["BAA10Y"].pct_change()

stocks2_df["Credit_spread_change"]= spread["Credit_spread"]
stocks2_df["Credit_spread_change"] = stocks2_df["Credit_spread_change"].fillna(stocks2_df["Credit_spread_change"].mean())
stocks2_df.dropna(inplace=True)

In [16]:

import statsmodels.api as sm # type: ignore

X = stocks2_df[["Rate_change","Credit_spread_change","SPY_ret"]]
X = sm.add_constant(X)

y = stocks2_df["TLT_ret"]

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                TLT_ret   R-squared:                       0.776
Model:                            OLS   Adj. R-squared:                  0.775
Method:                 Least Squares   F-statistic:                     1441.
Date:                Wed, 01 Apr 2026   Prob (F-statistic):               0.00
Time:                        09:29:09   Log-Likelihood:                 5688.8
No. Observations:                1255   AIC:                        -1.137e+04
Df Residuals:                    1251   BIC:                        -1.135e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    0.0002 

In [17]:

import statsmodels.api as sm # type: ignore

X = stocks2_df[["Rate_change","Credit_spread_change","SPY_ret"]]
X = sm.add_constant(X)

y = stocks2_df["LQD_ret"]

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                LQD_ret   R-squared:                       0.026
Model:                            OLS   Adj. R-squared:                  0.024
Method:                 Least Squares   F-statistic:                     11.26
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           2.74e-07
Time:                        09:29:09   Log-Likelihood:                 3921.4
No. Observations:                1255   AIC:                            -7835.
Df Residuals:                    1251   BIC:                            -7814.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    0.0006 

In [18]:

import statsmodels.api as sm # type: ignore

X = stocks2_df[["Rate_change","Credit_spread_change","SPY_ret"]]
X = sm.add_constant(X)

y = stocks2_df["SPY_ret"]

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                SPY_ret   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 3.135e+32
Date:                Wed, 01 Apr 2026   Prob (F-statistic):               0.00
Time:                        09:29:09   Log-Likelihood:                 47152.
No. Observations:                1255   AIC:                        -9.430e+04
Df Residuals:                    1251   BIC:                        -9.428e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                -1.082e-18 

In [19]:
stocks2_df[["Rate_change","Credit_spread_change","SPY_ret"]].cov()


,Rate_change,Credit_spread_change,SPY_ret
Rate_change,0.000479,-0.000113,-0.000185
Credit_spread_change,-0.000113,0.000249,0.000032
SPY_ret,-0.000185,0.000032,0.000102


In [20]:
betas = np.array([
    [-0.04,  0.01,  0.40],   # Treasury Bond   40%
    [ 0.04, -0.09,  0.16],   # IG Corp Bond    35%
    [ 0.00,  0.00,  1.00],   # S&P 500 ETF     25%
])

weights    = np.array([0.40, 0.35, 0.25])
port_value = 1_000_000
N_SIMS     = 50_000

sigma = np.array([
    [ 0.0004, -0.0001, -0.0002],
    [-0.0001,  0.0002,  0.0000],
    [-0.0002,  0.0000,  0.0016],
])
asset_names  = ["Treasury Bond", "IG Corp Bond", "S&P 500 ETF"]
factor_names = ["20Y Rate", "Credit Spread", "SPY Returns"]

# ─────────────────────────────────────────────────────────────────
# STEP 1 — CHOLESKY (no fixing needed)
# ─────────────────────────────────────────────────────────────────

print("\nSTEP 1 — Covariance matrix Σ:")
print(f"         20Y Rate   Credit    SPY")
labels = ["20Y Rate", "Credit  ", "SPY     "]
for i, row in enumerate(sigma):
    print(f"  {labels[i]}  " + "  ".join(f"{v:+.4f}" for v in row))

L = np.linalg.cholesky(sigma)

print("\nSTEP 2 — Cholesky L matrix (Σ = L × Lᵀ):")
for row in L:
    print("  " + "  ".join(f"{v:+.6f}" for v in row))
print(f"\n  Verify L @ Lᵀ == Σ: {np.allclose(L @ L.T, sigma)}")

# ─────────────────────────────────────────────────────────────────
# STEP 2 — SHOW 5000 SCENARIOS BY HAND
# ─────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("STEP 3 — First 5000 scenarios (step by step)")
print("=" * 60)
print((f"\n{'Scen':>4}  {'z1':>6} {'z2':>6} {'z3':>6}  "
      f"{'ε1 rate':>9} {'ε2 credit':>10} {'ε3 SPY':>8}  {'P&L ($)':>10}"))
print("-" * 75)

np.random.seed(42)
Z_all = np.random.standard_normal((3, N_SIMS))

for s in range(5000):
    z   = Z_all[:, s]
    eps = L @ z

    pnl = 0.0
    for a in range(3):
        r    = float(betas[a] @ eps)
        pnl += weights[a] * r * port_value

    print(f"{s+1:>4}  "
          f"{z[0]:>+6.2f} {z[1]:>+6.2f} {z[2]:>+6.2f}  "
          f"{eps[0]*100:>+8.3f}% {eps[1]*100:>+9.3f}% {eps[2]*100:>+7.3f}%  "
          f"{pnl:>+10,.0f}")

# ─────────────────────────────────────────────────────────────────
# STEP 3 — FULL MONTE CARLO
# ─────────────────────────────────────────────────────────────────

np.random.seed(42)
Z   = np.random.standard_normal((3, N_SIMS))
eps = L @ Z                        # (3 x N_SIMS) correlated shocks
ret = betas @ eps                  # (3 x N_SIMS) asset returns
pnl = (weights @ ret) * port_value # (N_SIMS,)   portfolio P&L

# ─────────────────────────────────────────────────────────────────
# STEP 4 — RISK METRICS
# ─────────────────────────────────────────────────────────────────

sorted_pnl = np.sort(pnl)
idx95      = int(0.05 * N_SIMS)
idx99      = int(0.01 * N_SIMS)

var95  = sorted_pnl[idx95]
cvar95 = sorted_pnl[:idx95].mean()
var99  = sorted_pnl[idx99]
cvar99 = sorted_pnl[:idx99].mean()

print("\n" + "=" * 60)
print("STEP 4 — RISK METRICS")
print("=" * 60)
print(f"\n  Portfolio value:          ${port_value:>12,}")
print(f"  Simulations:              {N_SIMS:>12,}")
print(f"\n  95% monthly VaR:         ${var95:>+12,.0f}")
print(f"  95% CVaR (Exp Shortfall): ${cvar95:>+12,.0f}")
print(f"\n  99% monthly VaR:         ${var99:>+12,.0f}")
print(f"  99% CVaR (Exp Shortfall): ${cvar99:>+12,.0f}")
print(f"\n  Mean P&L:                ${pnl.mean():>+12,.0f}")
print(f"  Std dev:                 ${pnl.std():>+12,.0f}")
print(f"  Worst scenario:          ${pnl.min():>+12,.0f}")
print(f"  Best scenario:           ${pnl.max():>+12,.0f}")

# ─────────────────────────────────────────────────────────────────
# STEP 5 — FACTOR DECOMPOSITION
# ─────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("STEP 5 — FACTOR RISK DECOMPOSITION (1σ shock each)")
print("=" * 60)
print(f"\n  {'Factor':<15} {'1σ shock':>10} {'Port impact':>14} {'% of total var':>16}")
print("  " + "-" * 58)

factor_stds = np.sqrt(np.diag(sigma))
total_var   = pnl.var()

for f in range(3):
    shock       = np.zeros(3)
    shock[f]    = factor_stds[f]
    r_single    = betas @ shock
    pnl_single  = float(weights @ r_single) * port_value
    contrib_pct = (pnl_single ** 2) / total_var * 100
    print(f"  {factor_names[f]:<15} {factor_stds[f]*100:>+9.3f}%  "
          f"${pnl_single:>+12,.0f}  {contrib_pct:>14.1f}%")


STEP 1 — Covariance matrix Σ:
         20Y Rate   Credit    SPY
  20Y Rate  +0.0004  -0.0001  -0.0002
  Credit    -0.0001  +0.0002  +0.0000
  SPY       -0.0002  +0.0000  +0.0016

STEP 2 — Cholesky L matrix (Σ = L × Lᵀ):
  +0.020000  +0.000000  +0.000000
  -0.005000  +0.013229  +0.000000
  -0.010000  -0.003780  +0.038545

  Verify L @ Lᵀ == Σ: True

STEP 3 — First 5000 scenarios (step by step)

Scen      z1     z2     z3    ε1 rate  ε2 credit   ε3 SPY     P&L ($)
---------------------------------------------------------------------------
   1   +0.50  +0.10  +1.03    +0.993%    -0.118%  +3.439%     +16,037
   2   -0.14  -0.06  -1.16    -0.277%    -0.016%  -4.291%     -19,985
   3   +0.65  +0.95  +0.58    +1.295%    +0.935%  +1.211%      +5,358
   4   +1.52  +1.53  -0.62    +3.046%    +1.266%  -4.489%     -21,329
   5   -0.23  +0.69  -0.33    -0.468%    +1.026%  -1.287%      -6,272
   6   -0.23  +1.01  +0.05    -0.468%    +1.455%  +0.035%        -226
   7   +1.58  -0.22  -0.12    +3.158